# Deploy Content Understanding Spoke

**Run this notebook once before the rest of this content understanding collection**

Deploys a dedicated `rg-foundry-cu-{suffix}` resource group with an AI Services
account (`aif-cu-{suffix}`), a `cu-project` Foundry project, local model deployments
for CU field extraction, and a `/cu` API on the core APIM.

## What gets deployed

| Resource | Type | Where |
|----------|------|-------|
| `rg-foundry-cu-{suffix}` | Resource Group | New — dedicated CU RG |
| `aif-cu-{suffix}` | Foundry Account (AI Services) | `rg-foundry-cu-{suffix}` — new |
| `gpt-4.1-mini` | Model Deployment | On `aif-cu-{suffix}` — new |
| `text-embedding-3-large` | Model Deployment | On `aif-cu-{suffix}` — new |
| `cu-project` | Foundry Project | Child of `aif-cu-{suffix}` — new |
| `landing-zone-apim` | Project connection (ApiManagement) | On `cu-project` — new |
| `content-understanding-api` | APIM API (`/cu`) | On `apim-foundry-{suffix}` in hub RG — new |
| `foundry-gateway-cu` | APIM Subscription | Scoped to `openai` API — new |
| RBAC assignments | Deployer + project MI + APIM MI | On `aif-cu-{suffix}` |

> **Dedicated resource group required.** Content Understanding needs local model
> deployments (`gpt-4.1-mini`, `text-embedding-3-large`) for field extraction analyzers.
> The `deny-model-deployments` policy must **not** be assigned to `rg-foundry-cu-{suffix}`.

## Sequence

```
10-00-deploy-setup   ← this notebook (Bicep deployment, .env outputs)
10-01-cu-analyze     ← list analyzers, analyze a document, poll results
```

## Prerequisites

1. **Lab 1A complete** — `GATEWAY_URL` must be in `.env`.
2. **Lab 1C complete** — `MULTI_ACCOUNT` must be in `.env` (confirms shared account exists).
3. **Python environment** — run `uv sync` from the repo root; select the `.venv` kernel.
4. **Azure CLI** — run `az login` before executing cells.

## Step 0: Prerequisites check

In [1]:
import os
import json
import subprocess
import base64
import time
import tempfile
from pathlib import Path
from IPython.display import clear_output
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

repo_root = Path(
    subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip()
)
env_file = repo_root / '.env'

# Read .env manually — same pattern as other deploy notebooks
with open(env_file) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ[key] = value

assert 'GATEWAY_URL' in os.environ, (
    'GATEWAY_URL not found in .env — complete Lab 1A first'
)
assert 'MULTI_ACCOUNT' in os.environ, (
    'MULTI_ACCOUNT not found in .env — complete Lab 1C first'
)

GATEWAY_URL   = os.environ['GATEWAY_URL']
CHAT_MODEL    = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')
BOOTSTRAP_KEY = os.environ['ALPHA_GATEWAY_KEY']

print(f'Gateway URL   : {GATEWAY_URL}')
print(f'Chat model    : {CHAT_MODEL}')
print(f'Bootstrap key : {BOOTSTRAP_KEY[:4]}... (hidden)')
print('Prerequisites: OK')

Gateway URL   : https://apim-foundry-6fe574.azure-api.net/openai
Chat model    : gpt-4.1-mini
Bootstrap key : a9fe... (hidden)
Prerequisites: OK


## Step 1: Resolve deployment context

In [2]:
# Get subscription ID
SUB_ID = subprocess.run(
    'az account show --query id -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()

# Derive APIM service name and suffix from GATEWAY_URL
# e.g. https://apim-foundry-{suffix}.azure-api.net/openai -> apim-foundry-{suffix}
APIM_NAME = GATEWAY_URL.split('//')[1].split('.')[0]
SUFFIX    = APIM_NAME.split('-')[-1]
CORE_RG    = f'rg-foundry-core-{SUFFIX}'
CU_RG     = f'rg-foundry-cu-{SUFFIX}'

# Get deployer principal ID from cached JWT — avoids a network call
token   = subprocess.run(
    'az account get-access-token --query accessToken -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()
padding = '=' * (4 - len(token.split('.')[1]) % 4)
PRINCIPAL_ID = json.loads(base64.b64decode(token.split('.')[1] + padding))['oid']

# Get APIM managed identity principal ID for rbac.bicep
APIM_PRINCIPAL_ID = subprocess.run(
    f'az apim show -g {CORE_RG} -n {APIM_NAME} --query identity.principalId -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()

print(f'Subscription  : {SUB_ID}')
print(f'APIM name     : {APIM_NAME}')
print(f'Suffix        : {SUFFIX}')
print(f'Core RG        : {CORE_RG}')
print(f'CU RG         : {CU_RG}')
print(f'Principal ID  : {PRINCIPAL_ID}')
print(f'APIM MI       : {APIM_PRINCIPAL_ID}')

Subscription  : 025aba94-0c4a-443d-8826-466477e2850f
APIM name     : apim-foundry-6fe574
Suffix        : 6fe574
Core RG        : rg-foundry-core-6fe574
CU RG         : rg-foundry-cu-6fe574
Principal ID  : 5db0aa5d-f281-47a3-9720-04727dec61e8
APIM MI       : b8094a38-b107-4ecf-8b9f-d3f27880e6b2


## Step 2: Deploy CU spoke Bicep

Creates `rg-foundry-cu-{suffix}` and deploys `main.bicep` into it.
Provisions `aif-cu-{suffix}`, model deployments, `cu-project`, and RBAC.
Takes ~5–8 minutes.

In [4]:
# Create resource group in the same region as the hub
HUB_LOCATION = subprocess.run(
    f'az group show -n {CORE_RG} --query location -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()
print(f'Hub location: {HUB_LOCATION}')

subprocess.run(
    ['az', 'group', 'create', '-n', CU_RG, '-l', HUB_LOCATION, '-o', 'none'],
    check=True
)
print(f'Resource group ready: {CU_RG}')

result = subprocess.run(
    [
        'az', 'deployment', 'group', 'create',
        '-g', CU_RG,
        '--template-file', 'main.bicep',
        '-p', f'deployerPrincipalId={PRINCIPAL_ID}',
        '-p', f'apimUrl={GATEWAY_URL}',
        '-p', f'apimSubscriptionKey={BOOTSTRAP_KEY}',
        '--name', 'cu-main',
        '-o', 'table',
    ],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Bicep deployment failed — see stderr above')

Hub location: eastus2
Resource group ready: rg-foundry-cu-6fe574
Name     State      Timestamp                         Mode         ResourceGroup
-------  ---------  --------------------------------  -----------  --------------------
cu-main  Succeeded  2026-03-09T19:38:28.767795+00:00  Incremental  rg-foundry-cu-6fe574



## Step 3: Read outputs and write to `.env`

In [5]:
r = subprocess.run(
    f'az deployment group show -g "{CU_RG}" -n cu-main --query properties.outputs -o json',
    shell=True, capture_output=True, text=True
)
if r.returncode != 0 or not r.stdout.strip():
    raise RuntimeError(f'Failed to read deployment outputs.\n{r.stderr}')

out = json.loads(r.stdout)

CU_ACCOUNT_NAME              = out['accountName']['value']
CU_ACCOUNT_ENDPOINT          = out['accountEndpoint']['value']
CU_FOUNDRY_PROJECT           = out['projectName']['value']
CU_FOUNDRY_PROJECT_ENDPOINT  = out['projectEndpoint']['value']
CU_APIM_CONNECTION           = out['apimConnectionName']['value']

print(f'Account name     : {CU_ACCOUNT_NAME}')
print(f'Account endpoint : {CU_ACCOUNT_ENDPOINT}')
print(f'Project name     : {CU_FOUNDRY_PROJECT}')
print(f'Project endpoint : {CU_FOUNDRY_PROJECT_ENDPOINT}')
print(f'APIM connection  : {CU_APIM_CONNECTION}')

# Merge into .env
existing = {}
for line in env_file.read_text().splitlines():
    if '=' in line and not line.startswith('#'):
        k, _, v = line.partition('=')
        existing[k.strip()] = v.strip()

existing.update({
    'CU_ACCOUNT_ENDPOINT':         CU_ACCOUNT_ENDPOINT,
    'CU_FOUNDRY_PROJECT':          CU_FOUNDRY_PROJECT,
    'CU_FOUNDRY_PROJECT_ENDPOINT': CU_FOUNDRY_PROJECT_ENDPOINT,
    'CU_APIM_CONNECTION':          CU_APIM_CONNECTION,
    'CU_RESOURCE_GROUP':           CU_RG,
})

env_file.write_text('\n'.join(f'{k}={v}' for k, v in existing.items()) + '\n')
print(f'\n.env updated: {env_file}')

Account name     : aif-cu-ii5drx
Account endpoint : https://aif-cu-ii5drx.cognitiveservices.azure.com/
Project name     : cu-project
Project endpoint : https://aif-cu-ii5drx.services.ai.azure.com/api/projects/cu-project
APIM connection  : landing-zone-apim

.env updated: /home/jp/developments/jp/msft/work/foundry-nextgen/.env


## Step 4: Deploy APIM API and RBAC

Deploys `apim-cu-api.bicep` into the core resource group (adds the `/cu` API with
7 operations and governance policy), then deploys `rbac.bicep` into the CU resource
group (grants APIM managed identity Cognitive Services User on `aif-cu-{suffix}`).

In [11]:
# Deploy /cu APIM API into hub RG
api_result = subprocess.run(
    [
        'az', 'deployment', 'group', 'create',
        '-g', CORE_RG,
        '--template-file', 'apim-cu-api.bicep',
        '-p', f'apimName={APIM_NAME}',
        '-p', f'cuEndpoint={CU_ACCOUNT_ENDPOINT}',
        '--name', 'cu-api',
        '-o', 'table',
    ],
    capture_output=True, text=True
)
print(api_result.stdout)
if api_result.returncode != 0:
    print(api_result.stderr)
    raise RuntimeError('APIM API deployment failed — see stderr above')

# Deploy RBAC into CU RG — grants APIM MI Cognitive Services User on aif-cu-{suffix}
rbac_result = subprocess.run(
    [
        'az', 'deployment', 'group', 'create',
        '-g', CU_RG,
        '--template-file', 'rbac.bicep',
        '-p', f'cuAccountName={CU_ACCOUNT_NAME}',
        '-p', f'apimPrincipalId={APIM_PRINCIPAL_ID}',
        '--name', 'cu-rbac',
        '-o', 'table',
    ],
    capture_output=True, text=True
)
print(rbac_result.stdout)
if rbac_result.returncode != 0:
    print(rbac_result.stderr)
    raise RuntimeError('RBAC deployment failed — see stderr above')

Name    State      Timestamp                         Mode         ResourceGroup
------  ---------  --------------------------------  -----------  ---------------------
cu-api  Succeeded  2026-03-09T19:44:10.780590+00:00  Incremental  rg-foundry-core-6fe574

Name     State      Timestamp                         Mode         ResourceGroup
-------  ---------  --------------------------------  -----------  --------------------
cu-rbac  Succeeded  2026-03-09T19:44:53.148068+00:00  Incremental  rg-foundry-cu-6fe574



## Step 5: Create dedicated APIM subscription

Creates `foundry-gateway-cu` subscription on the core APIM, scoped to the
`content-understanding-api` (deployed in Step 4). This gives the CU workload its own
rate-limit bucket for `/cu` calls.

The `landing-zone-apim` connection on `cu-project` retains the bootstrap key set
during Bicep deployment — it is used for OpenAI inference and must remain scoped
to the `openai` API.

> **If this step fails** (e.g. insufficient APIM permissions), run the fallback cell
> to use `ALPHA_GATEWAY_KEY` instead.

In [13]:
APIM_SUB_NAME = 'foundry-gateway-cu'
APIM_BASE_URI = (
    f'https://management.azure.com/subscriptions/{SUB_ID}'
    f'/resourceGroups/{CORE_RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}'
)

# Scope to content-understanding-api specifically (deployed in Step 4).
# A service-level scope is not supported by the APIM subscriptions API.
# The APIM connection on cu-project retains BOOTSTRAP_KEY (set in Bicep) for
# OpenAI inference — CU_GATEWAY_KEY is only used for /cu API calls.
create_result = subprocess.run(
    [
        'az', 'rest', '--method', 'PUT',
        '--uri', f'{APIM_BASE_URI}/subscriptions/{APIM_SUB_NAME}?api-version=2024-06-01-preview',
        '--body', json.dumps({
            'properties': {
                'displayName': 'Foundry CU Gateway Access',
                'scope': (
                    f'/subscriptions/{SUB_ID}/resourceGroups/{CORE_RG}'
                    f'/providers/Microsoft.ApiManagement/service/{APIM_NAME}'
                    f'/apis/content-understanding-api'
                ),
                'state': 'active',
            }
        }),
        '--headers', 'Content-Type=application/json',
    ],
    capture_output=True, text=True
)

if create_result.returncode != 0:
    print(f'APIM subscription creation failed:\n{create_result.stderr.strip()}')
    print('\nRun the fallback cell below to use ALPHA_GATEWAY_KEY instead.')
    CU_GATEWAY_KEY = None
else:
    CU_GATEWAY_KEY = subprocess.run(
        f'az rest --method POST'
        f' --uri "{APIM_BASE_URI}/subscriptions/{APIM_SUB_NAME}/listSecrets?api-version=2024-06-01-preview"'
        f' --query primaryKey -o tsv',
        shell=True, capture_output=True, text=True
    ).stdout.strip()
    print(f'APIM subscription created : {APIM_SUB_NAME}')
    print(f'CU gateway key            : {CU_GATEWAY_KEY[:4]}... (hidden)')

APIM subscription created : foundry-gateway-cu
CU gateway key            : 016a... (hidden)


In [ ]:
# Fallback: use ALPHA_GATEWAY_KEY if Step 5 failed.
# Uncomment and run only if needed.

# CU_GATEWAY_KEY = os.environ['ALPHA_GATEWAY_KEY']
# print(f'Using ALPHA_GATEWAY_KEY as CU_GATEWAY_KEY: {CU_GATEWAY_KEY[:4]}... (hidden)')

In [14]:
assert CU_GATEWAY_KEY, 'CU_GATEWAY_KEY is not set — check Step 5 or run the fallback cell.'

existing = {}
for line in env_file.read_text().splitlines():
    if '=' in line and not line.startswith('#'):
        k, _, v = line.partition('=')
        existing[k.strip()] = v.strip()

existing['CU_GATEWAY_KEY'] = CU_GATEWAY_KEY
env_file.write_text('\n'.join(f'{k}={v}' for k, v in existing.items()) + '\n')
print(f'CU_GATEWAY_KEY written to {env_file}')

CU_GATEWAY_KEY written to /home/jp/developments/jp/msft/work/foundry-nextgen/.env


## Step 6: Set CU model defaults

Patches the CU account's default model configuration via the APIM gateway to
register `gpt-4.1-mini` and `text-embedding-3-large` as the deployments used by
field extraction analyzers. The `modelDeployments` map uses model name as key and
deployment name as value.

In [ ]:
import urllib.request
import urllib.error

# Reload .env to pick up CU_GATEWAY_KEY if written in a prior cell
with open(env_file) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ[key] = value

CU_GATEWAY_KEY = os.environ['CU_GATEWAY_KEY']

# Derive the CU gateway base URL from GATEWAY_URL
# https://apim-foundry-{suffix}.azure-api.net/openai  ->  .../cu
CU_GATEWAY_URL = GATEWAY_URL.rstrip('/').rsplit('/openai', 1)[0] + '/cu'
CU_API_VERSION = '2025-11-01'

defaults_url = f'{CU_GATEWAY_URL}/defaults?api-version={CU_API_VERSION}'

# modelDeployments: model name -> deployment name
defaults_body = json.dumps({
    'modelDeployments': {
        'gpt-4.1-mini':           'gpt-4.1-mini',
        'text-embedding-3-large': 'text-embedding-3-large',
    }
}).encode('utf-8')

req = urllib.request.Request(
    defaults_url,
    data=defaults_body,
    method='PATCH',
    headers={
        'api-key':      CU_GATEWAY_KEY,
        'Content-Type': 'application/json',
    }
)

with urllib.request.urlopen(req) as resp:
    body = json.loads(resp.read())

assert body['modelDeployments']['gpt-4.1-mini'] == 'gpt-4.1-mini'
assert body['modelDeployments']['text-embedding-3-large'] == 'text-embedding-3-large'
print(f'CU defaults PATCH: OK')
print(json.dumps(body, indent=2))

## Step 7: Wait for RBAC propagation

Azure role assignments can take up to 2 minutes to propagate. Running the verification
or subsequent notebooks immediately may produce 403 errors.

In [23]:
for remaining in range(90, 0, -10):
    clear_output(wait=True)
    print(f'Waiting for RBAC to propagate... {remaining}s')
    time.sleep(10)

clear_output(wait=True)
print('RBAC propagation wait complete.')

RBAC propagation wait complete.


## Step 8: Verify deployment

In [24]:
credential     = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=CU_FOUNDRY_PROJECT_ENDPOINT, credential=credential)
connections    = list(project_client.connections.list())

print(f'Project   : {CU_FOUNDRY_PROJECT_ENDPOINT}')
print('Connections:')
for c in connections:
    d = dict(c)
    print(f'  {d.get("name")} ({d.get("type")}) -> {d.get("target")}')

connection_names = [dict(c).get('name') for c in connections]
assert 'landing-zone-apim' in connection_names, (
    f'Expected landing-zone-apim in connections, got: {connection_names}'
)

CU_GATEWAY_KEY = os.environ['CU_GATEWAY_KEY']
assert CU_GATEWAY_KEY, 'CU_GATEWAY_KEY is not set'

print(f'\nDeployment summary:')
print(f'  CU_RESOURCE_GROUP           : {CU_RG}')
print(f'  CU_FOUNDRY_PROJECT_ENDPOINT : {CU_FOUNDRY_PROJECT_ENDPOINT}')
print(f'  CU_GATEWAY_KEY              : {CU_GATEWAY_KEY[:8]}...')
print('Verification: OK')

Project   : https://aif-cu-ii5drx.services.ai.azure.com/api/projects/cu-project
Connections:
  landing-zone-apim (ApiManagement) -> https://apim-foundry-6fe574.azure-api.net/openai

Deployment summary:
  CU_RESOURCE_GROUP           : rg-foundry-cu-6fe574
  CU_FOUNDRY_PROJECT_ENDPOINT : https://aif-cu-ii5drx.services.ai.azure.com/api/projects/cu-project
  CU_GATEWAY_KEY              : 016ab9c4...
Verification: OK


## Done

The CU resources are deployed and `.env` has been updated.

**Keys written to `.env`:**

| Key | Description |
|-----|-------------|
| `CU_ACCOUNT_ENDPOINT` | CU account cognitive services endpoint |
| `CU_FOUNDRY_PROJECT` | Project name (`cu-project`) |
| `CU_FOUNDRY_PROJECT_ENDPOINT` | Project endpoint URL |
| `CU_APIM_CONNECTION` | APIM connection name on `cu-project` |
| `CU_GATEWAY_KEY` | Dedicated APIM subscription key for CU workload |
| `CU_RESOURCE_GROUP` | Resource group (`rg-foundry-cu-{suffix}`) |

**Next step:** run `10-01-cu-analyze.ipynb` to list analyzers, analyze a document,
and poll results via the governed APIM gateway.

---
## Troubleshooting

### IfMatchPreconditionFailed on redeploy

If you rerun Step 2 after Step 5 has already patched the APIM connection, ARM's
incremental mode will fail with an ETag mismatch. Delete the connection first, then
rerun Step 2.

In [ ]:
# Run only if redeployment fails with IfMatchPreconditionFailed, then re-run Step 2.

# connection_uri = (
#     f'https://management.azure.com/subscriptions/{SUB_ID}'
#     f'/resourceGroups/{CU_RG}/providers/Microsoft.CognitiveServices/accounts/{CU_ACCOUNT_NAME}'
#     f'/projects/{CU_FOUNDRY_PROJECT}/connections/{CU_APIM_CONNECTION}?api-version=2025-04-01-preview'
# )
# r = subprocess.run(
#     f'az rest --method DELETE --uri "{connection_uri}"',
#     shell=True, capture_output=True, text=True
# )
# print('Deleted' if r.returncode == 0 else r.stderr)